In [ ]:
# Install Tiny-SIMA and the optional VLM backend into this kernel.
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Leave room for the PyTorch VLM before importing JAX.
os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.35"),
)

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("Tiny-SIMA requires Python 3.11 or 3.12.")

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the Tiny-SIMA repository.")

cuda_override = os.environ.get("TINY_SIMA_USE_CUDA")
if cuda_override is None:
    has_cuda = (
        shutil.which("nvidia-smi") is not None
        and subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    )
else:
    has_cuda = cuda_override.lower() in {"1", "true", "yes"}

extras = "notebook,vlm,cuda12" if has_cuda else "notebook,vlm"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-cache-dir",
    "--disable-pip-version-check",
    "-e",
    f"{repo_root}[{extras}]",
])
# Import Torchvision before Transformers. Transformers caches optional
# dependency availability on first import.
torch = importlib.import_module("torch")
torchvision = importlib.import_module("torchvision")
transformers = importlib.import_module("transformers")
outlines = importlib.import_module("outlines")
transformers_utils = importlib.import_module("transformers.utils")

print(
    "PyTorch:", torch.__version__,
    "Torchvision:", torchvision.__version__,
    "Transformers:", transformers.__version__,
    "Outlines: installed",
)
if not transformers_utils.is_torchvision_available():
    raise RuntimeError(
        "Transformers cached Torchvision as unavailable. Restart this "
        "kernel, then run this installation cell before any other cell."
    )

source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print("Repository:", repo_root)
print("CUDA detected:", has_cuda)
print("JAX memory fraction:", os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"])


# Phase 4: hierarchical VLM planner

This notebook evaluates a frozen VLM at skill boundaries. The model supplies the Say term (objective usefulness), while symbolic preconditions supply the Can term. It never sees the private scripted reference plan. Start it in a fresh kernel so the JAX memory fraction is applied before JAX initializes.

The evaluation has two layers:

1. Controlled-prefix evaluation isolates planner decisions by letting the scripted executor advance the world after each prediction.
2. Closed-loop evaluation connects the planner to the learned stochastic executor and measures objective success.

In [ ]:
import time

import jax
import matplotlib.pyplot as plt

from tiny_sima.config import load_vlm_planner_config
from tiny_sima.envs.chains import make_skill_chain_environment
from tiny_sima.envs.craftax import render_state
from tiny_sima.evaluation import (
    WOOD_TO_STONE_CHAIN,
    evaluate_planner_prefixes,
    evaluate_skill_chain,
    learned_skill_executor_from_checkpoint,
    run_vlm_smoke_test,
)
from tiny_sima.goals import SKILL_IDS
from tiny_sima.planner import (
    PlannerContext,
    SayCanPlanner,
    SequentialUtilityScorer,
    TransformersVLMBackend,
    VLMUtilityScorer,
    build_planner_prompt,
    build_planner_system_prompt,
    constant_feasibility,
    constant_utility,
)

print("JAX devices:", jax.devices())


In [ ]:
CHECKPOINT = repo_root / "runs" / "skill-ppo-local-1m" / "checkpoints" / "final"
VLM_CONFIG_PATH = repo_root / "configs" / "vlm_planner_qwen3_2b.yaml"

QUICK_EVAL = True
RUN_VLM_SMOKE_TEST = True
RUN_CLOSED_LOOP = True
NUM_PREFIX_EPISODES = 1 if QUICK_EVAL else 8
NUM_CLOSED_LOOP_EPISODES = 2 if QUICK_EVAL else 16
SKILL_HORIZON = 64
EVALUATION_SEED = 51

vlm_config = load_vlm_planner_config(VLM_CONFIG_PATH)
print("VLM:", vlm_config.model_id)
print("Checkpoint exists:", CHECKPOINT.exists())


## Inspect the planner input

The VLM receives a rendered frame plus a compact text record. This cell is cheap and should be inspected before loading model weights.

In [ ]:
preview_environment = make_skill_chain_environment()
preview_state = preview_environment.reset(jax.random.PRNGKey(EVALUATION_SEED)).state
preview_prompt = build_planner_prompt(
    preview_state,
    context=PlannerContext(
        objective=WOOD_TO_STONE_CHAIN.objective,
    ),
    candidate_skills=SKILL_IDS,
)
preview_pixels = render_state(
    preview_state,
    vlm_config.environment_name,
    block_pixel_size=vlm_config.block_pixel_size,
)
plt.figure(figsize=(8, 6))
plt.imshow(preview_pixels)
plt.axis("off")
plt.show()
print("SYSTEM:\n", build_planner_system_prompt())
print("\nUSER:\n", preview_prompt)


## End-to-end VLM smoke test

This performs one real multimodal generation and verifies constrained decoding, complete-ranking validation, and symbolic grounding before the longer evaluations run.

In [ ]:
backend = TransformersVLMBackend(vlm_config)
vlm_utility = VLMUtilityScorer(backend=backend, config=vlm_config)

if RUN_VLM_SMOKE_TEST:
    smoke_result = run_vlm_smoke_test(
        vlm_config,
        scorer=vlm_utility,
        seed=EVALUATION_SEED,
    )
    print(smoke_result.to_dict())


## Controlled-prefix planner ablation

The same randomized states are used for all four planners. The VLM response cache means the VLM-only and VLM-plus-symbolic conditions share generations when their controlled states are identical.

In [ ]:
prefix_planners = {
    "scripted_oracle": SayCanPlanner(
        utility_scorer=SequentialUtilityScorer(WOOD_TO_STONE_CHAIN.reference_skills),
    ),
    "vlm_only": SayCanPlanner(
        utility_scorer=vlm_utility,
        feasibility_scorer=constant_feasibility,
    ),
    "feasibility_only": SayCanPlanner(utility_scorer=constant_utility),
    "vlm_plus_symbolic": SayCanPlanner(utility_scorer=vlm_utility),
}

prefix_results = {}
for name, planner in prefix_planners.items():
    started = time.perf_counter()
    result = evaluate_planner_prefixes(
        planner,
        num_episodes=NUM_PREFIX_EPISODES,
        seed=EVALUATION_SEED,
        skill_horizon=SKILL_HORIZON,
    )
    prefix_results[name] = result
    print(
        f"{name:20s} accepted={result['accepted_next_skill_accuracy']:.3f} "
        f"parse_valid={result['valid_utility_response_rate']:.3f} "
        f"valid={result['valid_selection_rate']:.3f} "
        f"grounded={result['grounded_selection_rate']:.3f} "
        f"seconds={time.perf_counter() - started:.2f}"
    )


In [ ]:
for trace in prefix_results["vlm_plus_symbolic"]["traces"]:
    selected = trace.selected_skill.value if trace.selected_skill else "none"
    accepted = ",".join(skill.value for skill in trace.accepted_skills)
    ranking = " > ".join(skill.value for skill in trace.utility_ranking)
    print(
        f"episode={trace.episode} boundary={trace.boundary} "
        f"reference={trace.reference_skill.value:20s} selected={selected:20s} "
        f"accepted=[{accepted}] matched={trace.matched} "
        f"parse_valid={trace.utility_valid} "
        f"feasibility={trace.selected_feasibility:.1f}"
    )
    if trace.utility_raw_response is not None:
        print(f"  ranking={ranking}")
        print(f"  raw={trace.utility_raw_response}")
    if trace.utility_error is not None:
        print(f"  error={trace.utility_error}")


## Closed-loop planner/executor ablation

This is intentionally small by default because VLM generation is sequential at option boundaries. Increase the episode count only after the controlled-prefix output is sensible.

In [ ]:
if RUN_CLOSED_LOOP:
    if not CHECKPOINT.exists():
        raise FileNotFoundError(f"Missing executor checkpoint: {CHECKPOINT}")
    executor = learned_skill_executor_from_checkpoint(
        CHECKPOINT,
        deterministic=False,
    )
    closed_loop_results = {}
    for name, planner in prefix_planners.items():
        started = time.perf_counter()
        result = evaluate_skill_chain(
            executor,
            planner=planner,
            num_episodes=NUM_CLOSED_LOOP_EPISODES,
            seed=EVALUATION_SEED,
            skill_horizon=SKILL_HORIZON,
        )
        closed_loop_results[name] = result
        print(
            f"{name:20s} success={result['success_rate']:.3f} "
            f"skills={result['mean_completed_skills']:.2f} "
            f"steps={result['mean_environment_steps']:.1f} "
            f"seconds={time.perf_counter() - started:.2f} "
            f"terminations={result['termination_counts']}"
        )


## Interpretation

- Low accepted-skill accuracy with valid JSON is a planner reasoning or visual-grounding failure.
- Invalid utility responses are reported separately from states with no viable skill.
- A large VLM-only to grounded improvement shows that symbolic affordances are preventing impossible requests.
- Good prefix accuracy but poor closed-loop success points to executor recovery or planner adaptation after failures.
- If prompting is weak, preserve these zero-shot numbers before considering demonstrations or parameter-efficient fine-tuning.